In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install datasets

In [ ]:
import pandas as pd
import urllib.request
import zipfile
import os

# Download directly from the official source
url = "https://www.cs.ucsb.edu/~william/data/liar_dataset.zip"
urllib.request.urlretrieve(url, "liar_dataset.zip")

# Unzip
with zipfile.ZipFile("liar_dataset.zip", "r") as zip_ref:
    zip_ref.extractall("liar_data")

print(os.listdir("liar_data"))

['README', 'train.tsv', 'valid.tsv', 'test.tsv']


In [ ]:
cols = ['id', 'label', 'statement', 'subject', 'speaker',
        'job', 'state', 'party', 'barely_true', 'false',
        'half_true', 'mostly_true', 'pants_fire', 'context']

train_df = pd.read_csv('liar_data/train.tsv', sep='\t', header=None, names=cols)
test_df = pd.read_csv('liar_data/test.tsv', sep='\t', header=None, names=cols)
val_df = pd.read_csv('liar_data/valid.tsv', sep='\t', header=None, names=cols)

print(f"Train: {train_df.shape}")
print(train_df['label'].value_counts())

Train: (10240, 14)
label
half-true      2114
false          1995
mostly-true    1962
true           1676
barely-true    1654
pants-fire      839
Name: count, dtype: int64


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

os.makedirs('/content/drive/MyDrive/fake-news', exist_ok=True)
train_df.to_csv('/content/drive/MyDrive/fake-news/liar_train.csv', index=False)
test_df.to_csv('/content/drive/MyDrive/fake-news/liar_test.csv', index=False)
val_df.to_csv('/content/drive/MyDrive/fake-news/liar_val.csv', index=False)
print("Saved to Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to Drive!


In [ ]:
print("=== DATASET OVERVIEW ===")
print(f"Train size: {train_df.shape[0]} samples")
print(f"Test size: {test_df.shape[0]} samples")
print(f"Val size: {val_df.shape[0]} samples")

print("\n=== LABEL DISTRIBUTION (Train) ===")
print(train_df['label'].value_counts())

print("\n=== SAMPLE STATEMENTS ===")
for label in train_df['label'].unique():
    sample = train_df[train_df['label'] == label]['statement'].iloc[0]
    print(f"\n[{label}]: {sample[:100]}...")

print("\n=== TEXT LENGTH STATS ===")
train_df['text_length'] = train_df['statement'].str.len()
print(train_df['text_length'].describe())

print("\n=== MISSING VALUES ===")
print(train_df.isnull().sum())

=== DATASET OVERVIEW ===
Train size: 10240 samples
Test size: 1267 samples
Val size: 1284 samples

=== LABEL DISTRIBUTION (Train) ===
label
half-true      2114
false          1995
mostly-true    1962
true           1676
barely-true    1654
pants-fire      839
Name: count, dtype: int64

=== SAMPLE STATEMENTS ===

[false]: Says the Annies List political group supports third-trimester abortions on demand....

[half-true]: When did the decline of coal start? It started when natural gas took off that started to begin in (P...

[mostly-true]: Hillary Clinton agrees with John McCain "by voting to give George Bush the benefit of the doubt on I...

[true]: The Chicago Bears have had more starting quarterbacks in the last 10 years than the total number of ...

[barely-true]: Jim Dunnam has not lived in the district he represents for years now....

[pants-fire]: In the case of a catastrophic event, the Atlanta-area offices of the Centers for Disease Control and...

=== TEXT LENGTH STATS ===
count

In [ ]:
# Keep only what you need
label_map = {
    'true': 'real',
    'mostly-true': 'real',
    'half-true': 'uncertain',
    'barely-true': 'uncertain',
    'false': 'fake',
    'pants-fire': 'fake'
}

def preprocess(df):
    df = df[['statement', 'label']].copy()
    df['label'] = df['label'].map(label_map)
    df = df.dropna()
    df['statement'] = df['statement'].str.strip().str.lower()
    return df

train_clean = preprocess(train_df)
test_clean = preprocess(test_df)
val_clean = preprocess(val_df)

print("=== AFTER PREPROCESSING ===")
print(f"Train: {train_clean.shape}")
print(train_clean['label'].value_counts())

# Save cleaned version
train_clean.to_csv('/content/drive/MyDrive/fake-news/train_clean.csv', index=False)
test_clean.to_csv('/content/drive/MyDrive/fake-news/test_clean.csv', index=False)
val_clean.to_csv('/content/drive/MyDrive/fake-news/val_clean.csv', index=False)

print("\nCleaned data saved!")

=== AFTER PREPROCESSING ===
Train: (10240, 2)
label
uncertain    3768
real         3638
fake         2834
Name: count, dtype: int64

Cleaned data saved!
